<a href="https://colab.research.google.com/github/chiltonjoseadol/SOLID/blob/main/Refatoracao_SOLID_Loja_Online.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Refatorando um Sistema Ruim com SOLID em Python
**Faculdade:** Senac

**Aluno:** Miquissene Visa

**Atividade:** Loja Online — análise de problemas de design e refatoração aplicando SOLID

**Conteúdo do notebook**
1. Código original (comentado com os problemas)
2. Etapa 1 — Identificação dos problemas
3. Etapa 2 — Tabela de violações SOLID
4. Etapa 3 — Código refatorado
5. Prova de que o comportamento externo é o mesmo
6. Extensibilidade (novo frete e novo pagamento sem alterar o código existente)
7. Como cada princípio foi aplicado

## 1. Código original (comentado com os problemas)

In [1]:
# Sistema Ruim: Loja Online  (versão original, com os problemas assinalados)

# [PROBLEMA GERAL - SRP] Esta "classe Deus" gerencia produtos, frete, desconto,
# pagamento, nota fiscal, e-mail e ainda imprime o resumo da compra.
# (Nome alterado para LojaOnlineRuim apenas para não conflitar com a versão refatorada)
class LojaOnlineRuim:
    def __init__(self):
        self.produtos = []   # [PROBLEMA] produtos são dicionários soltos, sem modelo próprio
        self.total = 0       # [PROBLEMA] total guardado como estado duplicado (pode ficar inconsistente)

    def adicionar_produto(self, nome, preco):
        self.produtos.append({"nome": nome, "preco": preco})
        self.total += preco

    # [PROBLEMA - SRP] Regra de frete misturada com a loja.
    # [PROBLEMA - OCP] Novo tipo de frete => é preciso EDITAR este if/elif.
    def calcular_frete(self, tipo_frete):
        if tipo_frete == "sedex":
            return 25.0
        elif tipo_frete == "pac":
            return 15.0
        elif tipo_frete == "retirada":
            return 0.0
        else:
            return 50.0

    # [PROBLEMA - SRP] Regra de desconto misturada com a loja.
    # [PROBLEMA - OCP] Novo tipo de cliente => é preciso EDITAR este if/elif.
    def calcular_desconto(self, tipo_cliente):
        if tipo_cliente == "comum":
            return self.total * 0.05
        elif tipo_cliente == "premium":
            return self.total * 0.10
        elif tipo_cliente == "vip":
            return self.total * 0.15
        return 0

    # [PROBLEMA - SRP] Pagamento dentro da loja.
    # [PROBLEMA - OCP] Nova forma de pagamento => é preciso EDITAR este if/elif.
    # [PROBLEMA - DIP] Depende de implementações concretas (strings + print), sem abstração.
    def pagar(self, tipo_pagamento):
        if tipo_pagamento == "pix":
            print("Pagamento realizado via PIX")
        elif tipo_pagamento == "cartao":
            print("Pagamento realizado via Cartão")
        elif tipo_pagamento == "boleto":
            print("Pagamento realizado via Boleto")
        else:
            print("Forma de pagamento inválida")

    # [PROBLEMA - SRP/DIP] Envio de e-mail fixo dentro da loja; não dá para trocar por SMS/WhatsApp.
    def enviar_email_confirmacao(self):
        print("Enviando e-mail de confirmação...")

    # [PROBLEMA - SRP/DIP] Nota fiscal fixa dentro da loja.
    def gerar_nota_fiscal(self):
        print("Gerando nota fiscal...")

    # [PROBLEMA - SRP] Orquestra tudo, calcula valores E formata/imprime o resumo.
    # [PROBLEMA - DIP] Chama diretamente os métodos concretos da própria classe.
    # [PROBLEMA - Alto acoplamento] Qualquer mudança em qualquer regra afeta esta classe.
    def finalizar_compra(self, tipo_cliente, tipo_frete, tipo_pagamento):
        desconto = self.calcular_desconto(tipo_cliente)
        frete = self.calcular_frete(tipo_frete)
        valor_final = self.total - desconto + frete

        print("Produtos:")
        for produto in self.produtos:
            print(produto["nome"], produto["preco"])

        print("Total:", self.total)
        print("Desconto:", desconto)
        print("Frete:", frete)
        print("Valor final:", valor_final)

        self.pagar(tipo_pagamento)
        self.gerar_nota_fiscal()
        self.enviar_email_confirmacao()

# Como executar
loja = LojaOnlineRuim()
loja.adicionar_produto("Notebook", 3500)
loja.adicionar_produto("Mouse", 120)
loja.finalizar_compra(
    tipo_cliente="premium",
    tipo_frete="sedex",
    tipo_pagamento="pix"
)

Produtos:
Notebook 3500
Mouse 120
Total: 3620
Desconto: 362.0
Frete: 25.0
Valor final: 3283.0
Pagamento realizado via PIX
Gerando nota fiscal...
Enviando e-mail de confirmação...


## 2. Etapa 1 — Identificar problemas

**Quais responsabilidades estão misturadas na classe `LojaOnline`?**
- Gerenciamento de produtos / carrinho (`adicionar_produto`, `total`)
- Cálculo de frete (`calcular_frete`)
- Cálculo de desconto (`calcular_desconto`)
- Processamento de pagamento (`pagar`)
- Emissão de nota fiscal (`gerar_nota_fiscal`)
- Notificação por e-mail (`enviar_email_confirmacao`)
- Orquestração do checkout e apresentação/impressão do resumo (`finalizar_compra`)

**Quais trechos violam o princípio da Responsabilidade Única (SRP)?**
- A classe inteira: tem 7 motivos diferentes para mudar.
- `calcular_frete`, `calcular_desconto`, `pagar`, `gerar_nota_fiscal` e `enviar_email_confirmacao`: são regras/serviços independentes dentro da loja.
- `finalizar_compra`: calcula, imprime e dispara pagamento, nota e e-mail ao mesmo tempo.

**Quais partes violam o princípio Aberto/Fechado (OCP)?**
- Os `if/elif` de `calcular_frete`, `calcular_desconto` e `pagar`: cada novo tipo de frete, de cliente ou de pagamento obriga a **modificar** código já existente e testado, em vez de **estender**.

**Existe dependência direta de implementações concretas?**
- Sim. `finalizar_compra` chama diretamente `pagar`, `gerar_nota_fiscal` e `enviar_email_confirmacao`, todos concretos e da mesma classe. Não há abstração (interface) nem injeção de dependências, então não dá para trocar o e-mail por SMS, ou o `print` por um gateway real, sem editar a classe (violação do **DIP**). Além disso, as regras são escolhidas por *strings* ("sedex", "pix"...), o que é frágil.

**Que mudanças futuras obrigariam alterar essa classe?**
- Novo tipo de frete (ex.: "expresso") ou mudança de preço do frete
- Novo tipo de cliente ou nova regra de desconto (ex.: desconto por cupom)
- Nova forma de pagamento (ex.: criptomoeda, PayPal) ou integração com gateway real
- Trocar notificação por e-mail para SMS/WhatsApp, ou enviar vários canais
- Mudar o formato da nota fiscal ou integrar com a SEFAZ
- Mudar o formato de exibição do resumo (ex.: gerar PDF, JSON ou tela web)
- Adicionar quantidade, estoque ou impostos aos produtos

## 3. Etapa 2 — Relacionar com SOLID

| # | Problema encontrado | Princípio violado | Justificativa |
|---|---|---|---|
| 1 | Classe faz muitas coisas | **SRP** | A classe gerencia produtos, frete, desconto, pagamento, nota fiscal, e-mail e impressão do resumo; tem vários motivos para mudar. |
| 2 | `calcular_frete` com `if/elif` por tipo | **OCP** | Adicionar um novo frete exige editar o método existente, com risco de quebrar os demais. |
| 3 | `calcular_desconto` com `if/elif` por tipo de cliente | **OCP** | Um novo perfil de cliente obriga a modificar a classe. |
| 4 | `pagar` com `if/elif` por forma de pagamento | **OCP** | Cada nova forma de pagamento exige alterar o método central. |
| 5 | `finalizar_compra` chama métodos concretos da própria classe (pagar, nota fiscal, e-mail) | **DIP** | O módulo de alto nível (checkout) depende de detalhes de baixo nível, sem abstrações nem injeção de dependência. |
| 6 | Regras de negócio distintas (frete, desconto, pagamento) agrupadas num só tipo | **ISP** | Quem só precisa de frete ou só de pagamento é forçado a depender de uma classe enorme com métodos que não usa; faltam interfaces pequenas e específicas. |
| 7 | Seleção de comportamento por *strings* e retorno "padrão" embutido nos `else` | **LSP** (risco) | Não existe hierarquia/contrato; ao extrair as regras para classes, todas precisam ser substituíveis entre si (mesmo método, mesmo tipo de retorno, sem surpresas). Na refatoração isso é garantido. |
| 8 | Total guardado em `self.total` separado da lista de produtos | Coesão / encapsulamento | Dois estados que precisam andar juntos podem ficar inconsistentes; o total deveria ser derivado dos produtos. |

## 4. Etapa 3 — Código refatorado

In [6]:
from abc import ABC, abstractmethod
from dataclasses import dataclass


# ======================================================================
# MODELO: produto e carrinho  (responsabilidade: guardar produtos e total)
# ======================================================================
@dataclass(frozen=True)
class Produto:
    nome: str
    preco: float


class Carrinho:
    def __init__(self):
        self._produtos = []

    def adicionar(self, nome, preco):
        self._produtos.append(Produto(nome, preco))

    @property
    def produtos(self):
        return tuple(self._produtos)

    @property
    def total(self):
        # total derivado dos produtos: não existe estado duplicado
        return sum(p.preco for p in self._produtos)


# ======================================================================
# REGISTRO GENÉRICO  (permite adicionar novas opções sem editar código existente)
# ======================================================================
class Registro:
    """Associa um nome (ex.: 'pix') a uma estratégia, com um padrão para nomes desconhecidos."""

    def __init__(self, padrao):
        self._itens = {}
        self._padrao = padrao

    def registrar(self, nome, estrategia):
        self._itens[nome] = estrategia

    def obter(self, nome):
        return self._itens.get(nome, self._padrao)


# ======================================================================
# FRETE  (OCP + LSP + ISP)
# ======================================================================
class Frete(ABC):
    @abstractmethod
    def calcular(self, carrinho) -> float:
        ...


class FreteSedex(Frete):
    def calcular(self, carrinho):
        return 25.0


class FretePac(Frete):
    def calcular(self, carrinho):
        return 15.0


class FreteRetirada(Frete):
    def calcular(self, carrinho):
        return 0.0


class FretePadrao(Frete):
    """Comportamento original para tipo de frete desconhecido."""
    def calcular(self, carrinho):
        return 50.0


# ======================================================================
# DESCONTO  (OCP + LSP + ISP)
# ======================================================================
class PoliticaDesconto(ABC):
    @abstractmethod
    def calcular(self, total) -> float:
        ...


class DescontoPercentual(PoliticaDesconto):
    def __init__(self, taxa):
        self._taxa = taxa

    def calcular(self, total):
        return total * self._taxa


class SemDesconto(PoliticaDesconto):
    def calcular(self, total):
        return 0


# ======================================================================
# PAGAMENTO  (OCP + LSP + ISP)
# ======================================================================
class MetodoPagamento(ABC):
    @abstractmethod
    def pagar(self, valor) -> None:
        ...


class PagamentoPix(MetodoPagamento):
    def pagar(self, valor):
        print("Pagamento realizado via PIX")


class PagamentoCartao(MetodoPagamento):
    def pagar(self, valor):
        print("Pagamento realizado via Cartão")


class PagamentoBoleto(MetodoPagamento):
    def pagar(self, valor):
        print("Pagamento realizado via Boleto")


class PagamentoInvalido(MetodoPagamento):
    """Comportamento original para forma de pagamento desconhecida."""
    def pagar(self, valor):
        print("Forma de pagamento inválida")


# ======================================================================
# NOTA FISCAL E NOTIFICAÇÃO  (abstrações pequenas + implementações)
# ======================================================================
class EmissorNotaFiscal(ABC):
    @abstractmethod
    def emitir(self, carrinho) -> None:
        ...


class NotaFiscalConsole(EmissorNotaFiscal):
    def emitir(self, carrinho):
        print("Gerando nota fiscal...")


class Notificador(ABC):
    @abstractmethod
    def notificar(self, mensagem=None) -> None:
        ...


class NotificadorEmail(Notificador):
    def notificar(self, mensagem=None):
        print("Enviando e-mail de confirmação...")


# ======================================================================
# APRESENTAÇÃO  (responsabilidade: formatar/imprimir o resumo)
# ======================================================================
class ResumoCompraConsole:
    def exibir(self, carrinho, desconto, frete, valor_final):
        print("Produtos:")
        for produto in carrinho.produtos:
            print(produto.nome, produto.preco)

        print("Total:", carrinho.total)
        print("Desconto:", desconto)
        print("Frete:", frete)
        print("Valor final:", valor_final)


# ======================================================================
# SERVIÇO DE CHECKOUT  (DIP: recebe abstrações por injeção de dependência)
# ======================================================================
class ServicoCheckout:
    def __init__(self, fretes, descontos, pagamentos,
                 emissor_nf, notificador, resumo):
        self._fretes = fretes
        self._descontos = descontos
        self._pagamentos = pagamentos
        self._emissor_nf = emissor_nf
        self._notificador = notificador
        self._resumo = resumo

    def finalizar(self, carrinho, tipo_cliente, tipo_frete, tipo_pagamento):
        desconto = self._descontos.obter(tipo_cliente).calcular(carrinho.total)
        frete = self._fretes.obter(tipo_frete).calcular(carrinho)
        valor_final = carrinho.total - desconto + frete

        self._resumo.exibir(carrinho, desconto, frete, valor_final)

        self._pagamentos.obter(tipo_pagamento).pagar(valor_final)
        self._emissor_nf.emitir(carrinho)
        self._notificador.notificar()


# ======================================================================
# COMPOSIÇÃO (único lugar que conhece as classes concretas)
# ======================================================================
def criar_fretes():
    registro = Registro(padrao=FretePadrao())
    registro.registrar("sedex", FreteSedex())
    registro.registrar("pac", FretePac())
    registro.registrar("retirada", FreteRetirada())
    return registro


def criar_descontos():
    registro = Registro(padrao=SemDesconto())
    registro.registrar("comum", DescontoPercentual(0.05))
    registro.registrar("premium", DescontoPercentual(0.10))
    registro.registrar("vip", DescontoPercentual(0.15))
    return registro


def criar_pagamentos():
    registro = Registro(padrao=PagamentoInvalido())
    registro.registrar("pix", PagamentoPix())
    registro.registrar("cartao", PagamentoCartao())
    registro.registrar("boleto", PagamentoBoleto())
    return registro


# Fachada com a MESMA interface pública da classe original,
# para que o comportamento externo continue idêntico.
class LojaOnline:
    def __init__(self, checkout=None):
        self._carrinho = Carrinho()
        self._checkout = checkout or ServicoCheckout(
            fretes=criar_fretes(),
            descontos=criar_descontos(),
            pagamentos=criar_pagamentos(),
            emissor_nf=NotaFiscalConsole(),
            notificador=NotificadorEmail(),
            resumo=ResumoCompraConsole(),
        )

    def adicionar_produto(self, nome, preco):
        self._carrinho.adicionar(nome, preco)

    def finalizar_compra(self, tipo_cliente, tipo_frete, tipo_pagamento):
        self._checkout.finalizar(
            self._carrinho, tipo_cliente, tipo_frete, tipo_pagamento
        )


# Como executar (idêntico ao original)
loja = LojaOnline()
loja.adicionar_produto("Notebook", 3500)
loja.adicionar_produto("Mouse", 120)
loja.finalizar_compra(
    tipo_cliente="premium",
    tipo_frete="sedex",
    tipo_pagamento="pix"
)

Produtos:
Notebook 3500
Mouse 120
Total: 3620
Desconto: 362.0
Frete: 25.0
Valor final: 3283.0
Pagamento realizado via PIX
Gerando nota fiscal...
Enviando e-mail de confirmação...


## 5. Mesmo comportamento externo

A saída acima deve ser exatamente a do sistema original:

```
Produtos:
Notebook 3500
Mouse 120
Total: 3620
Desconto: 362.0
Frete: 25.0
Valor final: 3283.0
Pagamento realizado via PIX
Gerando nota fiscal...
Enviando e-mail de confirmação...
```

Os casos de borda também foram preservados: frete desconhecido continua custando 50.0, cliente desconhecido continua sem desconto e pagamento desconhecido continua exibindo "Forma de pagamento inválida".

In [3]:
# Casos de borda (mesmos resultados do sistema original)
loja2 = LojaOnline()
loja2.adicionar_produto("Teclado", 200)
loja2.finalizar_compra(tipo_cliente="desconhecido", tipo_frete="drone", tipo_pagamento="fiado")
# Esperado: Desconto: 0 | Frete: 50.0 | Valor final: 250.0 | "Forma de pagamento inválida"

Produtos:
Teclado 200
Total: 200
Desconto: 0
Frete: 50.0
Valor final: 250.0
Forma de pagamento inválida
Gerando nota fiscal...
Enviando e-mail de confirmação...


In [4]:
# Comparação automática: original (LojaOnlineRuim) x refatorada (LojaOnline)
import io, contextlib

def executar(classe, tipo_cliente, tipo_frete, tipo_pagamento):
    saida = io.StringIO()
    with contextlib.redirect_stdout(saida):
        l = classe()
        l.adicionar_produto("A", 10.5)
        l.adicionar_produto("B", 99)
        l.finalizar_compra(tipo_cliente, tipo_frete, tipo_pagamento)
    return saida.getvalue()

iguais = True
for cliente in ["comum", "premium", "vip", "outro"]:
    for frete in ["sedex", "pac", "retirada", "outro"]:
        for pagamento in ["pix", "cartao", "boleto", "outro"]:
            if executar(LojaOnlineRuim, cliente, frete, pagamento) != executar(LojaOnline, cliente, frete, pagamento):
                iguais = False

print("Mesmo comportamento nas 64 combinações:", iguais)

Mesmo comportamento nas 64 combinações: True


## 6. Extensibilidade: novos tipos sem alterar o código existente

Abaixo adicionamos um **frete expresso**, uma **forma de pagamento por criptomoeda** e um **perfil de cliente "estudante"**, apenas criando classes novas e registrando-as. Nenhuma classe anterior (`ServicoCheckout`, `LojaOnline`, `Carrinho`...) foi modificada.

In [5]:
class FreteExpresso(Frete):
    def calcular(self, carrinho):
        return 40.0


class PagamentoCripto(MetodoPagamento):
    def pagar(self, valor):
        print("Pagamento realizado via Criptomoeda")


fretes = criar_fretes()
fretes.registrar("expresso", FreteExpresso())

pagamentos = criar_pagamentos()
pagamentos.registrar("cripto", PagamentoCripto())

descontos = criar_descontos()
descontos.registrar("estudante", DescontoPercentual(0.20))

checkout = ServicoCheckout(
    fretes=fretes,
    descontos=descontos,
    pagamentos=pagamentos,
    emissor_nf=NotaFiscalConsole(),
    notificador=NotificadorEmail(),
    resumo=ResumoCompraConsole(),
)

loja3 = LojaOnline(checkout=checkout)
loja3.adicionar_produto("Livro", 100)
loja3.finalizar_compra("estudante", "expresso", "cripto")

Produtos:
Livro 100
Total: 100
Desconto: 20.0
Frete: 40.0
Valor final: 120.0
Pagamento realizado via Criptomoeda
Gerando nota fiscal...
Enviando e-mail de confirmação...


## 7. Como cada princípio foi aplicado

**S — Responsabilidade Única (SRP)**
A classe `LojaOnline` foi dividida: `Carrinho` cuida dos produtos e do total; cada regra de frete, desconto e pagamento virou sua própria classe; `NotaFiscalConsole` e `NotificadorEmail` cuidam de nota e notificação; `ResumoCompraConsole` cuida da exibição; `ServicoCheckout` apenas orquestra o fluxo. Cada classe tem um único motivo para mudar.

**O — Aberto/Fechado (OCP)**
Os `if/elif` foram substituídos por estratégias (`Frete`, `PoliticaDesconto`, `MetodoPagamento`) com registro. Para criar um novo frete ou pagamento basta criar uma classe e registrá-la (seção 6), sem modificar o `ServicoCheckout` nem as regras existentes.

**L — Substituição de Liskov (LSP)**
Todas as subclasses respeitam o contrato da abstração: qualquer `Frete` pode ser usado onde se espera um `Frete`, com o mesmo método e o mesmo tipo de retorno. Até os casos padrão (`FretePadrao`, `SemDesconto`, `PagamentoInvalido`) seguem o contrato, então o checkout nunca precisa testar o tipo concreto.

**I — Segregação de Interfaces (ISP)**
Em vez de uma interface grande, existem interfaces pequenas, cada uma com um único método (`Frete.calcular`, `PoliticaDesconto.calcular`, `MetodoPagamento.pagar`, `EmissorNotaFiscal.emitir`, `Notificador.notificar`). Cada classe implementa apenas o que realmente usa.

**D — Inversão de Dependência (DIP)**
`ServicoCheckout` (alto nível) depende apenas de abstrações, recebidas pelo construtor (injeção de dependência). As classes concretas só aparecem na composição (`criar_fretes`, `criar_pagamentos`, `LojaOnline`), então é possível trocar o e-mail por SMS, ou o pagamento simulado por um gateway real, sem tocar no checkout, e também testar o serviço com objetos falsos (mocks).

**Resultados:** menor acoplamento, maior coesão, manutenção e expansão mais simples, e mesmo comportamento externo do sistema original.